# Můžu letět? Skutečné zprávy METAR a vítr ve výšce letu

[![Otevřít v Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenSkyLab-space/notebooks/blob/main/drone_weather_brno_cs.ipynb)

**OpenSkyLab · notebook k vizuálu** [Můžu letět? Počasí pro drony](https://openskylab.space/edu_viz_drone_weather.html) · [English version](drone_weather_brno_en.ipynb)

Ve vizuálu jsi dekódoval ukázkové zprávy METAR, přepočítal vítr z letiště na místo a výšku letu a rozhodoval podle semaforu. Tady stáhneš skutečné zprávy METAR z českých letišť za posledních 24 hodin, sám je dekóduješ a výsledek porovnáš s modelovou předpovědí větru ve 80 a 120 m.

**Cíl:** dekódovat METAR, spočítat relativní vlhkost a odhad základny oblačnosti, přepočítat vítr v 10 m na výšku letu nad jiným terénem, odhadnout návrat proti větru a rozhodnout o letu na příštích 24 hodin.

**Data:** zprávy METAR a TAF z letišť Brno-Tuřany (LKTB), Náměšť nad Oslavou (LKNA), Kunovice (LKKU) a Ostrava (LKMT) ze služby NOAA Aviation Weather Center; hodinová předpověď z Open-Meteo pro Brno-Bystrc u Brněnské přehrady. Data se mění každou půlhodinu, tvoje čísla budou jiná než v uloženém notebooku.

**Úkoly:** pět. Odpovědi piš do buněk *Tvoje odpověď* pod úkoly. Řádky označené `# ◀ parametr` můžeš měnit a buňku spustit znovu (Shift + Enter).

**Čas:** asi 45 min. Potřebuješ Google účet (Colab); nic se neinstaluje do počítače. Buňky spouštěj popořadě shora dolů.

## 1. Příprava a skutečné zprávy METAR

In [ ]:
import re, time, requests
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

def open_meteo(params, tries=6):
    """Dotaz na Open-Meteo; při chybě nebo vyčerpaném limitu to zkusí znovu."""
    for i in range(tries):
        try:
            r = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=60)
            if r.ok:
                return r.json()
            reason = r.json().get("reason", r.status_code)
        except requests.RequestException as e:
            reason = type(e).__name__
        print("Open-Meteo:", reason, "– zkusím znovu"); time.sleep(5 * (i + 1))
    raise RuntimeError("Open-Meteo neodpovídá, spusť buňku za chvíli znovu.")

In [ ]:
STATIONS = "LKTB,LKNA,LKKU,LKMT"          # ◀ parametr: kódy letišť ICAO
raw = requests.get("https://aviationweather.gov/api/data/metar", params={"ids": STATIONS, "format": "raw", "hours": 24}, timeout=60).text.strip().splitlines()
taf = requests.get("https://aviationweather.gov/api/data/taf", params={"ids": "LKTB", "format": "raw"}, timeout=60).text.strip()
print(f"{len(raw)} zpráv; nejnovější:"); print("\n".join(raw[:6])); print("\nTAF Brno:\n" + taf)

## 2. Vlastní dekodér

Zpráva má pevné pořadí skupin: letiště, čas, vítr (směr, rychlost, nárazy G), dohlednost v metrech nebo CAVOK, jevy (RA déšť, SN sníh, BR kouřmo, FG mlha, - slabý, + silný), oblačnost (FEW, SCT, BKN, OVC a výška ve stovkách stop), teplota/rosný bod (M = minus) a tlak QNH. Část za TEMPO, BECMG nebo NOSIG je krátká předpověď a dekodér ji vynechá.

In [ ]:
KT = 0.514444
def decode(m):
    body = re.split(r"\b(TEMPO|BECMG|NOSIG|RMK)\b", m)[0]
    d = {"stanice": body.split()[1], "čas": re.search(r"\b(\d{6})Z\b", body).group(1)}
    w = re.search(r"\b(\d{3}|VRB)(\d{2,3})(?:G(\d{2,3}))?(KT|MPS)\b", body)
    if w:
        f = KT if w.group(4) == "KT" else 1.0
        d["směr"] = w.group(1); d["vítr m/s"] = int(w.group(2)) * f; d["nárazy m/s"] = int(w.group(3)) * f if w.group(3) else np.nan
    if "CAVOK" in body:
        d["dohlednost m"] = 10000
    else:
        v = re.search(r"(?<=KT )(\d{4})\b|(?<=MPS )(\d{4})\b|\s(\d{4})\s", body)
        d["dohlednost m"] = int(next(g for g in v.groups() if g)) if v else np.nan
    d["jevy"] = " ".join(re.findall(r"(?<=\s)([-+]?(?:VC)?(?:MI|BC|PR|DR|BL|SH|TS|FZ)?(?:DZ|RA|SN|SG|PL|GR|GS|BR|FG|FU|HZ|SQ)+)(?=\s)", body + " "))
    clouds = re.findall(r"\b(FEW|SCT|BKN|OVC|VV)(\d{3})", body)
    d["oblačnost"] = " ".join(f"{c}{int(h) * 100}ft" for c, h in clouds)
    ceil = [int(h) * 100 * 0.3048 for c, h in clouds if c in ("BKN", "OVC", "VV")]
    d["základna BKN/OVC m"] = min(ceil) if ceil else np.nan
    tt = re.search(r"\b(M?\d{2})/(M?\d{2})\b", body)
    if tt:
        d["T °C"], d["Td °C"] = (int(x.replace("M", "-")) for x in tt.groups())
    return d

met = pd.DataFrame([decode(m) for m in raw])
met["RH %"] = 100 * np.exp(17.62 * met["Td °C"] / (243.12 + met["Td °C"])) / np.exp(17.62 * met["T °C"] / (243.12 + met["T °C"]))
met["odhad základny m"] = 125 * (met["T °C"] - met["Td °C"])
met.head(12).round(1)

> **✏️ Úkol 1.** Vyber si jednu zprávu a dekóduj ji ručně skupinu po skupině; souhlasí s tabulkou? Které skupiny by samy o sobě vyloučily let VLOS (dohlednost, mlha, nízká oblačnost, srážky)? Proč je pro pilota důležitější rozdíl T − Td než samotná teplota?

*Tvoje odpověď:* 

In [ ]:
c = met.dropna(subset=["základna BKN/OVC m"])
plt.figure(figsize=(5, 4.5))
plt.scatter(c["odhad základny m"], c["základna BKN/OVC m"])
plt.axline((0, 0), slope=1, color="gray", ls=":")
plt.xlabel("125 m · (T − Td)"); plt.ylabel("nahlášená základna BKN/OVC (m)"); plt.title(f"{len(c)} zpráv s oblačností BKN/OVC"); plt.show()

> **✏️ Úkol 2.** Jak dobře odhad 125 m · (T − Td) odpovídá skutečně nahlášené základně? Odhad platí pro kupovitou oblačnost v dobře promíchané vrstvě za dne. Najdi zprávy, kde selhává, a vysvětli proč (noc, vrstevnatá oblačnost, inverze, oblačnost z jiné vzduchové hmoty).

*Tvoje odpověď:* 

## 3. Vítr ve výšce letu nad jiným terénem

Letiště měří vítr v 10 m nad trávou (délka drsnosti z0 = 0,03 m). Ve výšce promíchání 60 m je vliv povrchu malý, tam vítr přeneseme na místo letu s jinou drsností z0 a posunutím nulové roviny d (Wieringa 1986). Platí pro neutrální zvrstvení (vítr, oblačno); za slunečného dne nebo v noci se profil liší.

In [ ]:
def wind_at(z, u10, z0=0.03, d=0.0, zb=60.0):
    ub = u10 * np.log(zb / 0.03) / np.log(10 / 0.03)                    # vítr ve výšce promíchání nad letištěm
    z = np.asarray(z, float)
    site = ub * np.log(np.maximum(z - d, z0 * 1.01) / z0) / np.log((zb - d) / z0)
    above = ub * np.log(z / 0.03) / np.log(zb / 0.03)
    return np.where(z <= zb, site, above)

TERRAIN = {"voda": (0.0002, 0), "pole, tráva": (0.03, 0), "předměstí": (0.5, 6), "les 25 m": (2.5, 17)}
u10 = float(met[met["stanice"] == "LKTB"]["vítr m/s"].iloc[0])     # ◀ parametr: vítr v 10 m (výchozí je poslední METAR z Brna)
z = np.linspace(1, 150, 300)
plt.figure(figsize=(6, 4.5))
for name, (z0, d) in TERRAIN.items():
    plt.plot(wind_at(z, u10, z0, d), z, label=f"{name} (z0 = {z0} m, d = {d} m)")
plt.axhline(60, color="gray", ls=":"); plt.xlabel("průměrný vítr (m/s)"); plt.ylabel("výška nad terénem (m)")
plt.title(f"vítr v 10 m na letišti {u10:.1f} m/s"); plt.legend(fontsize=8); plt.show()
for name, (z0, d) in TERRAIN.items():
    print(f"{name:12s}: 30 m {wind_at(30, u10, z0, d):4.1f} m/s, 100 m {wind_at(100, u10, z0, d):4.1f} m/s")

In [ ]:
fc = open_meteo({"latitude": 49.2467, "longitude": 16.4798, "forecast_days": 2, "wind_speed_unit": "ms", "timezone": "Europe/Prague",
                 "hourly": "wind_speed_10m,wind_speed_80m,wind_speed_120m,wind_gusts_10m,temperature_2m,dew_point_2m,visibility,precipitation"})
fc = pd.DataFrame(fc["hourly"]).set_index("time")
ratio80 = (fc.wind_speed_80m / fc.wind_speed_10m)[fc.wind_speed_10m > 2]
print(f"Open-Meteo pro Brno-Bystrc: poměr větru 80 m / 10 m medián {ratio80.median():.2f} (rozsah {ratio80.min():.2f}–{ratio80.max():.2f})")
print(f"náš profil nad polem: {wind_at(80, 1.0, 0.03, 0) / 1.0:.2f}, nad předměstím: {wind_at(80, 1.0, 0.5, 6) / wind_at(10, 1.0, 0.5, 6):.2f}")

> **✏️ Úkol 3.** Proč nad lesem vychází ve 30 m slabší průměrný vítr než nad vodou, ale ve 100 m skoro stejný? Kolega říká: „Letiště hlásí 6 m/s, dron snese 12 m/s, máme dvojnásobnou rezervu.“ Nastav `u10 = 6` a odpověz mu čísly pro výšku 100 m. Jak se poměr 80 m / 10 m z modelu Open-Meteo liší od našeho profilu a proč (zvrstvení během dne a noci)?

*Tvoje odpověď:* 

## 4. Cesta zpět proti větru

Dron letí vůči vzduchu rychlostí V, vítr ve výšce letu má rychlost U. Na trase D tam po větru a zpět proti větru trvá let t = D/(V + U) + D/(V − U).

In [ ]:
V, D = 12.0, 800          # ◀ parametr: rychlost vůči vzduchu (m/s) a vzdálenost od pilota (m)
U = np.linspace(0, 11.5, 200)
t_rt = D / (V + U) + D / (V - U)
plt.figure(figsize=(7, 3.5)); plt.plot(U, t_rt / 60); plt.axhline(2 * D / V / 60 * 2, color="gray", ls=":", label="dvojnásobek času bez větru")
plt.xlabel("vítr ve výšce letu U (m/s)"); plt.ylabel("tam a zpět (min)"); plt.legend(); plt.show()
print(f"bez větru {2 * D / V / 60:.1f} min; dvojnásobek nastane při U = {V / np.sqrt(2):.1f} m/s")

> **✏️ Úkol 4.** Odvoď ze vzorce, při jakém větru se čas tam a zpět zdvojnásobí (výsledek U = V/√2 vypisuje buňka). Proč nestačí, že průměrný vítr ve výšce letu je pod odolností dronu, když jsou nárazy (sloupec nárazy v METAR) o 30–50 % vyšší? Jak bys podle toho naplánoval trasu?

*Tvoje odpověď:* 

## 5. Semafor na příštích 48 hodin

Pro každou hodinu předpovědi z Open-Meteo pro místo letu rozhodneme: vítr ve 120 m a nárazy proti limitu dronu, srážky, dohlednost a riziko námrazy (teplota pod 1 °C a T − Td do 2 °C).

In [ ]:
limit = 12.0               # ◀ parametr: odolnost dronu vůči větru (m/s)
gust_factor = (fc.wind_gusts_10m / fc.wind_speed_10m.clip(lower=1)).clip(upper=2.5)
wind_fl = fc.wind_speed_120m
gust_fl = wind_fl * gust_factor
red = (gust_fl > limit) | (fc.precipitation > 0.5) | (fc.visibility < 1000)
icing = (fc.temperature_2m < 1) & (fc.temperature_2m - fc.dew_point_2m <= 2)
yellow = ~red & ((gust_fl > 0.75 * limit) | (fc.precipitation > 0) | (fc.visibility < 5000) | icing)
status = np.where(red, "NELETĚT", np.where(yellow, "pozor", "ok"))
out = pd.DataFrame({"vítr 120 m": wind_fl, "nárazy ~": gust_fl, "srážky mm": fc.precipitation, "dohlednost km": fc.visibility / 1000,
                    "T": fc.temperature_2m, "T−Td": fc.temperature_2m - fc.dew_point_2m, "verdikt": status}).round(1)
colors = {"ok": "#cfe8cf", "pozor": "#ffe7a3", "NELETĚT": "#f4b6b6"}
out.style.apply(lambda r: [f"background-color: {colors[r['verdikt']]}"] * len(r), axis=1)

> **✏️ Úkol 5.** Kdy bys v příštích 48 hodinách letěl a co v ostatních hodinách rozhoduje? Porovnej předpověď s posledním TAF pro Brno: shodují se? Nárazy ve 120 m jsme odhadli poměrem nárazů a větru v 10 m; proč je to hrubé a proč je výsledek spíš na straně bezpečnosti, nebo naopak? Nastav `limit = 10.7` (odolnost malého dronu do 250 g, např. DJI Mini 4 Pro). Co se změní?

*Tvoje odpověď:* 

> **✏️ Rozšíření (nepovinné).** Spolužáci mohou mít notebook pro jiné místo a jiná letiště (Roztoky a Praha; Brno-Bystrc a Brno-Tuřany; Hluboká a Linec). Porovnejte ve stejnou hodinu vítr z METAR, poměr větru 80 m / 10 m z Open-Meteo a výsledek semaforu. Proč se poměr liší podle místa a terénu?

*Tvoje odpověď:* 

---
### Zdroje dat a literatura

- METAR a TAF: NOAA/NWS Aviation Weather Center, https://aviationweather.gov/data/api/ (veřejná data vlády USA).
- Počasí: Open-Meteo.com (CC BY 4.0), https://open-meteo.com
- Relativní vlhkost a tlak nasycených par (Magnusův vzorec): WMO (2018). *Guide to Instruments and Methods of Observation* (WMO-No. 8), příloha 4.B.
- Přenos větru přes výšku promíchání 60 m: Wieringa, J. (1986). Roughness-dependent geographical interpolation of surface wind speed averages. *Quarterly Journal of the Royal Meteorological Society* 112, 867–889; Verkaik, J. W. (2000). Evaluation of two gustiness models for exposure correction calculations. *Journal of Applied Meteorology* 39, 1613–1626.
- Klíč ke zprávám METAR: ICAO Annex 3 / WMO-No. 306, kód FM 15.
- Odolnost DJI Mavic 3 Enterprise vůči větru 12 m/s, max. rychlost 15 m/s v normálním režimu, 21 m/s ve sportovním (technický list DJI); odolnost DJI Mini 4 Pro vůči větru 10,7 m/s.

### Jak citovat

OpenSkyLab (2026). Můžu letět? Skutečné zprávy METAR a vítr ve výšce letu [notebook, výukový materiál], verze 1.0. https://github.com/OpenSkyLab-space/notebooks/blob/main/drone_weather_brno_cs.ipynb. Licence CC BY 4.0.

Text a kód notebooku: OpenSkyLab, licence [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Data mají licence svých poskytovatelů (viz výše).

*Materiál připravil tým OpenSkyLab s pomocí generativní AI Claude (Anthropic), která pomáhala s návrhem kódu a textu. Za obsah odpovídají autoři.*